# 🧪 Sprint 2: Agent Evaluation Workbench

This workbench is for **Real-World AI Engineering**. We move beyond vibes and start measuring how our agent handles different scenarios, phases, and personas.

## Goals:
1. **A/B Test Prompts**: Compare how different phase instructions affect behavior.
2. **Scenario Simulation**: Run the agent against "Golden Scenarios" (e.g., Honeymooners, Budget Solo).
3. **State Integrity**: Verify the agent updates the Rich State (`TripShape`, `MentalModel`) correctly.

In [ ]:
import os
import json
import sys
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Add the api directory to path so we can import our actual code
sys.path.append(os.path.abspath("../services/api"))

from agent.models import VacationPlan, VacationPlanPatch
from agent.orchestrator import AgentOrchestrator
from agent.prompt import SystemPrompts

# Load environment variables
load_dotenv("../.env")
print("✅ Workbench Initialized")

## 1. Scenario Runner
We define personas to see if the agent adapts its funnel (Discovery vs. Proposal) correctly.

In [ ]:
SCENARIOS = [
    {
        "name": "The Vague Dreamer",
        "initial_message": "I just need to get away. Somewhere beautiful."
    },
    {
        "name": "The Goal-Oriented Planner",
        "initial_message": "Planning a 10-day honeymoon in Italy from NYC in June. Budget is $8k."
    },
    {
        "name": "The Budget Backpacker",
        "initial_message": "Looking for the cheapest possible beach vacation for me and a friend."
    }
]

def run_scenario(scenario_idx: int, turns: int = 3):
    scenario = SCENARIOS[scenario_idx]
    print(f"=== Testing Scenario: {scenario['name']} ===")
    
    orchestrator = AgentOrchestrator()
    plan = VacationPlan()
    history = []
    
    current_input = scenario["initial_message"]
    
    for i in range(turns):
        print(f"\n[Turn {i+1}] 👤 User: {current_input}")
        history.append({"role": "user", "content": current_input})
        
        response, plan, new_msgs = orchestrator.run_turn(history, plan)
        history.extend(new_msgs)
        history.append({"role": "assistant", "content": response})
        
        print(f"🤖 Agent: {response}")
        print(f"📊 State: PHASE={plan.phase} | UNKNOWNS={plan.mental_model.unknowns}")
        
        # For simulation, we'll just ask for more detail on turn 2
        if i == 0:
            current_input = "I'm thinking of Europe, maybe?"
        else:
            current_input = "What do you suggest?"

run_scenario(1) # Run the Goal-Oriented Planner

## 2. A/B Prompt Testing
Compare how the agent behaves with different SHARED_GUIDELINES.

In [ ]:
def test_prompt_variation(variant_guidelines: str, user_input: str):
    # Temporarily override guidelines
    original = SystemPrompts.SHARED_GUIDELINES
    SystemPrompts.SHARED_GUIDELINES = variant_guidelines
    
    orchestrator = AgentOrchestrator()
    plan = VacationPlan()
    response, plan, _ = orchestrator.run_turn([{"role": "user", "content": user_input}], plan)
    
    print(f"--- GUIDELINES VARIANT ---\n{response}\n")
    
    SystemPrompts.SHARED_GUIDELINES = original

# Compare 'Strict Interrogator' vs 'Friendly Guide'
test_prompt_variation(
    "1. Be very formal. 2. Ask 5 questions at once. 3. Don't use tools unless essential.", 
    "I want to go to Tokyo"
)